In [1]:
!pip install transformers datasets seqeval accelerate evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=47ee4d0f6aaca58431c7bdeb2ac7314a290b1a82b653e0c240ddede2517eeb8d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


# **Data Preprocessing**

In [15]:
import pandas as pd
import re
from datasets import Dataset
from transformers import AutoTokenizer

# -----------------------
# Parameters
# -----------------------
# CamelBERT-Mix is ideal for health forums and clinical notes
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
MAX_LENGTH = 128

# Load your dataset
df = pd.read_csv('Full_NER_Dataset.csv')

# -----------------------
# 1️⃣ Improved Arabic Normalization
# -----------------------
def normalize_arabic(text):
    if not isinstance(text, str): return ""
    text = re.sub("[إأآ]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    # Remove Arabic diacritics (Tashkeel)
    text = re.sub(r"[\u064B-\u0652]", "", text)
    return text.strip()

df = df[df['Token'].notna()].copy()
df['Token'] = df['Token'].apply(normalize_arabic)
df = df[df['Token'] != ""].copy()

# -----------------------
# 2️⃣ Dynamic Label Mapping
# -----------------------
# This captures all categories: DISEASE, SYMPTOM, MEDICATION, PROCEDURE, BODY_PART
label_list = sorted(df['Entity tag'].unique().tolist())
if 'O' in label_list:
    label_list.remove('O')
    label_list = ['O'] + label_list  # Ensure 'O' is at index 0

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

print(f"Detected Categories: {label_list}")

# -----------------------
# 3️⃣ Split into Sentences (Grouping Tokens)
# -----------------------
sentences = []
sentence_tokens = []
sentence_labels = []

# Clinical notes often use commas or periods as boundaries
punctuation = {'.', '!', '؟', '،', ','}

for token, label in zip(df['Token'], df['Entity tag']):
    sentence_tokens.append(token)
    sentence_labels.append(label)
    if token in punctuation:
        sentences.append({
            'tokens': sentence_tokens,
            'ner_tags': [label_to_id[l] for l in sentence_labels]
        })
        sentence_tokens = []
        sentence_labels = []

# Catch any remaining tokens
if sentence_tokens:
    sentences.append({
        'tokens': sentence_tokens,
        'ner_tags': [label_to_id[l] for l in sentence_labels]
    })

# -----------------------
# 4️⃣ Create & Split Dataset
# -----------------------
full_ds = Dataset.from_list(sentences)
ds_split = full_ds.train_test_split(test_size=0.2, seed=42)
train_val = ds_split['train'].train_test_split(test_size=0.1, seed=42)

train_ds = train_val['train']
val_ds = train_val['test']
test_ds = ds_split['test']

# -----------------------
# 5️⃣ Tokenize & Align (Crucial for Transformers)
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens (like [CLS]) get -100
            if word_idx is None:
                label_ids.append(-100)
            # Label only the first sub-token of a word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Following sub-tokens get -100 to ignore them in loss calculation
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Run Tokenization
train_tokenized = train_ds.map(tokenize_and_align_labels, batched=True)
val_tokenized = val_ds.map(tokenize_and_align_labels, batched=True)
test_tokenized = test_ds.map(tokenize_and_align_labels, batched=True)

print(f"✅ Preprocessing complete.")
print(f"Final Count -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Detected Categories: ['O', 'B-BODY_PART', 'B-DISEASE', 'B-MEDICATION', 'B-PROCEDURE', 'B-SYMPTOM', 'I-BODY_PART', 'I-DISEASE', 'I-MEDICATION', 'I-PROCEDURE', 'I-SYMPTOM']


Map:   0%|          | 0/7329 [00:00<?, ? examples/s]

Map:   0%|          | 0/815 [00:00<?, ? examples/s]

Map:   0%|          | 0/2036 [00:00<?, ? examples/s]

✅ Preprocessing complete.
Final Count -> Train: 7329, Val: 815, Test: 2036


In [16]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# **Training Loop**

In [17]:
from transformers import EarlyStoppingCallback
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
import evaluate
from seqeval.metrics import classification_report

# 1. Load Metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 2. Initialize Model
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

training_args = TrainingArguments(
    output_dir="./results/best_f1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # keep best LR
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,         # 🔥 increase epochs
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


# 5. EXECUTE TRAINING
print("🚀 Training started...")
trainer.train()
trainer.save_model("./best_arabic_clinical_ner_model")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-853372885.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Training started...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.120728,0.792000,0.765217,0.778378,0.959809
2,0.184800,0.124937,0.770308,0.797101,0.783476,0.958255
3,0.086800,0.115438,0.782772,0.807729,0.795055,0.962585
4,0.059900,0.126387,0.807205,0.800966,0.804074,0.963806
5,0.045200,0.132855,0.818812,0.799034,0.808802,0.966137
6,0.036500,0.146198,0.815686,0.803865,0.809732,0.964139
7,0.026200,0.148229,0.816103,0.793237,0.804508,0.964694
8,0.019900,0.157516,0.813609,0.797101,0.805271,0.964805


# **Overall Test Evaluation**

In [18]:
# Level 1: The "Scoreboard" - Quick summary of performance on unseen data
print("--- 1. Final Results on Test Dataset ---")
test_results = trainer.evaluate(test_tokenized)

print(f"Overall Precision: {test_results['eval_precision']:.4f}")
print(f"Overall Recall:    {test_results['eval_recall']:.4f}")
print(f"Overall F1-Score:  {test_results['eval_f1']:.4f}")

--- 1. Final Results on Test Dataset ---


Overall Precision: 0.8477
Overall Recall:    0.8398
Overall F1-Score:  0.8438


# **Category-Wise Evaluation**

In [6]:
from seqeval.metrics import classification_report
import numpy as np

def get_detailed_report(dataset, name="Final Test Set"):
    # 1. Get predictions from the model
    output = trainer.predict(dataset)
    predictions = np.argmax(output.predictions, axis=2)
    labels = output.label_ids

    # 2. Clean up predictions: Remove -100 (special tokens) and map IDs to Labels
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 3. Print the clinical category report
    print(f"\n--- 2. Clinical Category Report ({name}) ---")
    print(classification_report(true_labels, true_predictions))

# Run it
get_detailed_report(test_tokenized)


--- 2. Clinical Category Report (Final Test Set) ---
              precision    recall  f1-score   support

   BODY_PART       0.88      0.85      0.87       530
     DISEASE       0.88      0.89      0.89       322
  MEDICATION       0.81      0.83      0.82       361
   PROCEDURE       0.86      0.79      0.82       403
     SYMPTOM       0.79      0.82      0.81       763

   micro avg       0.84      0.83      0.84      2379
   macro avg       0.84      0.84      0.84      2379
weighted avg       0.84      0.83      0.84      2379



# **Linguistic Evaluation (MSA vs. Dialect)**

In [7]:
import torch

def predict_tokens(tokens_list):
    # Tokenize the input tokens
    tokenized_inputs_batch = tokenizer(
        tokens_list,
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Get word_ids BEFORE moving to device, as BatchEncoding object is lost after conversion to dict
    word_ids = tokenized_inputs_batch.word_ids(batch_index=0)

    # Move input tensors to the same device as the model
    device = model.device
    tokenized_inputs_on_device = {k: v.to(device) for k, v in tokenized_inputs_batch.items()}

    # Make predictions
    with torch.no_grad():
        outputs = model(**tokenized_inputs_on_device)
    predictions = outputs.logits.argmax(dim=-1).squeeze().tolist()

    # Align predictions with original tokens, ignoring special tokens and subwords
    aligned_predictions = []
    previous_word_idx = None
    for i, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            aligned_predictions.append(id_to_label[predictions[i]])
        previous_word_idx = word_idx
    return aligned_predictions

def evaluate_subset(tokens_list, true_labels_list=None):
    preds = predict_tokens(tokens_list)
    print("\nSubset Inference:")
    for t, l in zip(tokens_list, preds):
        print(f"{t}\t{l}")

    if true_labels_list is not None:
        # The metric expects a list of lists for predictions and references
        results = metric.compute(
            predictions=[preds],
            references=[true_labels_list]
        )
        print("\nSubset Metrics:")
        print(f"Precision: {results['overall_precision']:.4f}")
        print(f"Recall:    {results['overall_recall']:.4f}")
        print(f"F1-score:  {results['overall_f1']:.4f}")

# -------------------------------
# 15. MSA & Dialectal Subset Testing
# -------------------------------
test_tokens_msa = [
    "يشعر", "المريض", "بـ", "حمى", "و", "تناول", "بانادول", "."
]

true_labels_msa = [
    "O", "O", "O", "B-SYMPTOM", "O", "O", "B-MEDICATION", "O"
]


# Dialectal Sentence: "My head hurts since yesterday and my eye is red, I took drops for the allergy."
test_tokens_dialect = [
    "رأسي", "بيوجعني", "من", "امبارح", "وعيني", "حمرا", "،",
    "خدت", "قطرة", "عشان", "الحساسية", "."
]

true_labels_dialect = [
    "B-BODY_PART", "O", "O", "O", "B-BODY_PART", "B-SYMPTOM", "O",
    "O", "B-MEDICATION", "O", "B-DISEASE", "O"
]

# Execute updated evaluations
evaluate_subset(test_tokens_msa, true_labels_msa)
evaluate_subset(test_tokens_dialect, true_labels_dialect)



Subset Inference:
يشعر	O
المريض	O
بـ	O
حمى	B-SYMPTOM
و	O
تناول	O
بانادول	B-MEDICATION
.	O

Subset Metrics:
Precision: 1.0000
Recall:    1.0000
F1-score:  1.0000

Subset Inference:
رأسي	B-BODY_PART
بيوجعني	O
من	O
امبارح	O
وعيني	O
حمرا	O
،	O
خدت	O
قطرة	B-MEDICATION
عشان	O
الحساسية	O
.	O

Subset Metrics:
Precision: 1.0000
Recall:    0.4000
F1-score:  0.5714


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

def run_experiment(
    model_name,
    learning_rate,
    epochs,
    freeze_layers=False,
    run_name="experiment"
):
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        id2label=id_to_label,
        label2id=label_to_id
    )

    # Optional layer freezing
    if freeze_layers:
        for param in model.bert.embeddings.parameters():
            param.requires_grad = False
        for layer in model.bert.encoder.layer[:6]:
            for param in layer.parameters():
                param.requires_grad = False

    training_args = TrainingArguments(
        output_dir=f"./results/{run_name}",
        eval_strategy="epoch",          # ✅ FIXED
        save_strategy="no",
        learning_rate=learning_rate,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=20,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()
    results = trainer.evaluate(test_tokenized)

    return {
        "Model": model_name.split("/")[-1],
        "LR": learning_rate,
        "Epochs": epochs,
        "Frozen": freeze_layers,
        "Precision": results["eval_precision"],
        "Recall": results["eval_recall"],
        "F1": results["eval_f1"]
    }


In [12]:
experiment_results = []

# Baseline
experiment_results.append(
    run_experiment(
        MODEL_NAME, 2e-5, 5,
        freeze_layers=False,
        run_name="baseline"
    )
)

# LR tuning
experiment_results.append(
    run_experiment(
        MODEL_NAME, 1e-5, 5,
        freeze_layers=False,
        run_name="lr_1e5"
    )
)

experiment_results.append(
    run_experiment(
        MODEL_NAME, 3e-5, 5,
        freeze_layers=False,
        run_name="lr_3e5"
    )
)

# Layer freezing
experiment_results.append(
    run_experiment(
        MODEL_NAME, 1e-5, 5,
        freeze_layers=True,
        run_name="frozen_layers"
    )
)


Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-131742812.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.145100,0.118992,0.797980,0.763285,0.780247,0.960586
2,0.070100,0.117468,0.785849,0.804831,0.795227,0.960586
3,0.078700,0.110622,0.805985,0.806763,0.806374,0.964694
4,0.057500,0.119023,0.816568,0.800000,0.808199,0.964250
5,0.042600,0.120298,0.829000,0.800966,0.814742,0.965693


Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-131742812.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.165300,0.132572,0.792929,0.758454,0.775309,0.958810
2,0.080400,0.127895,0.765683,0.801932,0.783388,0.958366
3,0.100700,0.122209,0.774859,0.798068,0.786292,0.960031
4,0.076100,0.120578,0.800584,0.795169,0.797867,0.962141
5,0.066900,0.122094,0.818363,0.792271,0.805106,0.962363


Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-131742812.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.144400,0.117662,0.809619,0.780676,0.794884,0.961141
2,0.065700,0.120171,0.795821,0.809662,0.802682,0.962585
3,0.070200,0.115086,0.805395,0.807729,0.806561,0.963362
4,0.047900,0.120094,0.813976,0.799034,0.806436,0.964028
5,0.033100,0.124894,0.820158,0.801932,0.810943,0.964583


Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-131742812.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.172800,0.143180,0.761668,0.741063,0.751224,0.955146
2,0.090300,0.129476,0.750466,0.778744,0.764343,0.956589
3,0.120200,0.120531,0.777883,0.795169,0.786431,0.959476
4,0.107500,0.119359,0.775763,0.785507,0.780605,0.959143
5,0.091800,0.116185,0.809852,0.794203,0.801951,0.962585


In [13]:
import pandas as pd

results_df = pd.DataFrame(experiment_results)
results_df


,Model,LR,Epochs,Frozen,Precision,Recall,F1
0,bert-base-arabic-camelbert-mix,0.00002,5,False,0.846550,0.830181,0.838285
1,bert-base-arabic-camelbert-mix,0.00001,5,False,0.846352,0.828920,0.837545
2,bert-base-arabic-camelbert-mix,0.00003,5,False,0.843537,0.833964,0.838723
3,bert-base-arabic-camelbert-mix,0.00001,5,True,0.834330,0.821354,0.827791


In [8]:
from collections import Counter
import torch

# Count labels from original dataframe
label_counts = Counter(df["Entity tag"])

total_tokens = sum(label_counts.values())
num_labels = len(label_list)

class_weights = []
for label in label_list:
    # Avoid division by zero
    count = label_counts.get(label, 1)
    weight = total_tokens / (num_labels * count)
    class_weights.append(weight)

class_weights = torch.tensor(class_weights)

print("Class Weights:")
for label, weight in zip(label_list, class_weights):
    print(f"{label:15s} -> {weight:.2f}")


Class Weights:
O               -> 0.11
B-BODY_PART     -> 3.64
B-DISEASE       -> 5.90
B-MEDICATION    -> 5.25
B-PROCEDURE     -> 4.95
B-SYMPTOM       -> 2.50
I-BODY_PART     -> 14.58
I-DISEASE       -> 5.03
I-MEDICATION    -> 10.42
I-PROCEDURE     -> 5.08
I-SYMPTOM       -> 4.96


In [11]:
from transformers import Trainer

class WeightedNERTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )

        loss = loss_fct(
            logits.view(-1, logits.size(-1)),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [12]:
from transformers import TrainingArguments, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

training_args = TrainingArguments(
    output_dir="./results/weighted_loss",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # BEST LR
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = WeightedNERTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()


Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2135211201.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedNERTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedNERTrainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.317800,0.308486,0.628800,0.759420,0.687965,0.933052
2,0.150400,0.284765,0.582666,0.772947,0.664452,0.920395
3,0.163100,0.290723,0.683782,0.810628,0.741821,0.944377
4,0.096600,0.290790,0.722747,0.813527,0.765455,0.951149
5,0.090800,0.316573,0.726334,0.815459,0.768320,0.952926


TrainOutput(global_step=2295, training_loss=0.20246493229419318, metrics={'train_runtime': 394.7122, 'train_samples_per_second': 92.84, 'train_steps_per_second': 5.814, 'total_flos': 628022734608936.0, 'train_loss': 0.20246493229419318, 'epoch': 5.0})

In [13]:
weighted_results = trainer.evaluate(test_tokenized)

weighted_results


{'eval_loss': 0.3138671815395355,
 'eval_precision': 0.7490594431903687,
 'eval_recall': 0.8369062631357713,
 'eval_f1': 0.790549930514195,
 'eval_accuracy': 0.9596009526805375,
 'eval_runtime': 5.12,
 'eval_samples_per_second': 397.659,
 'eval_steps_per_second': 25.0,
 'epoch': 5.0}